# LLM agent – CA_1, FOODS_3

Goal: a planner asks questions in plain language; the model chooses which of my functions to call, and every number in the answer comes from those functions.

## Toy: one tool, one loop

In [4]:
import json
import sys
from pathlib import Path

import anthropic
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
load_dotenv(ROOT / ".env")

from src.forecasting.forecast import forecast

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-haiku-5-5"

In [5]:
def get_forecast_total(item: str, days: int = 7) -> dict:
    """Total median forecast (q50) for one item over the next `days` days."""
    try:
        fc = forecast(item, "CA_1", horizon=days)
    except (KeyError, ValueError) as e:
        return {"error": str(e)}
    return {
        "item": item,
        "from": str(fc["date"].iloc[0].date()),
        "to": str(fc["date"].iloc[-1].date()),
        "days": days,
        "q50_total_units": round(float(fc["q50"].sum()), 1),
    }


get_forecast_total("FOODS_3_090", 7)

{'item': 'FOODS_3_090',
 'from': '2016-05-23',
 'to': '2016-05-29',
 'days': 7,
 'q50_total_units': 368.6}

In [6]:
TOOL = {
    "name": "get_forecast_total",
    "description": (
        "Total forecast demand (median, q50) for one FOODS_3 item in store CA_1, "
        "over the next 1-28 days after the data ends (2016-05-22). "
        "Use it for any question about how much an item will sell."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "item": {"type": "string", "description": "Item id, e.g. FOODS_3_090"},
            "days": {"type": "integer", "minimum": 1, "maximum": 28},
        },
        "required": ["item"],
    },
}

In [7]:
def run_toy(question: str, verbose: bool = True) -> str:
    """Ask, run any tool the model asks for, send the result back, repeat until it answers."""
    messages = [{"role": "user", "content": question}]
    for step in range(5):  # safety limit
        response = client.messages.create(
            model=MODEL, max_tokens=1024, tools=[TOOL], messages=messages
        )
        # Keep the whole reply, thinking blocks included: the API needs them back
        messages.append({"role": "assistant", "content": response.content})
        if verbose:
            print(f"step {step}: stop_reason = {response.stop_reason}")

        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text")

        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = get_forecast_total(**block.input)
                if verbose:
                    print(f"  tool call: {block.name}({block.input}) -> {output}")
                results.append(
                    {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)}
                )
        messages.append({"role": "user", "content": results})
    return "Stopped: too many tool calls."

In [8]:
answer = run_toy("Quantas unidades de FOODS_3_090 vão vender na próxima semana?")
print("\nAnswer:", answer)

step 0: stop_reason = tool_use
  tool call: get_forecast_total({'item': 'FOODS_3_090', 'days': 7}) -> {'item': 'FOODS_3_090', 'from': '2016-05-23', 'to': '2016-05-29', 'days': 7, 'q50_total_units': 368.6}
step 1: stop_reason = end_turn

Answer: Para a próxima semana (de 23 a 29 de maio de 2016), a previsão mediana para o item FOODS_3_090 na loja CA_1 é de cerca de **369 unidades** (368,6).


In [9]:
print(run_toy("How much will FOODS_9_999 sell in the next 10 days?"))
print("\n" + "=" * 60 + "\n")
print(run_toy("Qual vai ser a procura de FOODS_3_090 nos próximos 40 dias?"))
print("\n" + "=" * 60 + "\n")
print(run_toy("Qual é o preço de compra do FOODS_3_090 ao fornecedor?"))

step 0: stop_reason = tool_use
  tool call: get_forecast_total({'item': 'FOODS_9_999', 'days': 10}) -> {'error': '"No forecast for item \'FOODS_9_999\' in store \'CA_1\'"'}
step 1: stop_reason = end_turn
I can't forecast FOODS_9_999. The system has no forecast for that item in store CA_1, so the ID may be wrong or the item may not be sold there.

Item IDs for this forecast follow the FOODS_3 pattern, such as FOODS_3_090. If you send me the correct ID, I'll get the 10-day forecast for it.


step 0: stop_reason = tool_use
  tool call: get_forecast_total({'item': 'FOODS_3_090', 'days': 28}) -> {'item': 'FOODS_3_090', 'from': '2016-05-23', 'to': '2016-06-19', 'days': 28, 'q50_total_units': 1580.5}
step 1: stop_reason = end_turn
Não consigo dar a previsão completa de 40 dias. A ferramenta só permite previsões até 28 dias, por isso calculei apenas esse período.

Para FOODS_3_090 na loja CA_1, entre 23 de maio e 19 de junho de 2016 (28 dias), a procura mediana prevista é de **cerca de 1.580 u

**What the toy shows**

- The model picks the tool and its inputs (it chose `days=7` for "next week"); my code runs it and returns the result.
- Errors returned as data let it recover: an unknown item is explained, 40 days is cut to the 28-day limit.
- With no tool for the question (purchase price), it declines instead of guessing.
- But numbers it works out itself can be wrong: for the 12 days after 2016-06-19 it wrote "20 to 31 June", a date that does not exist. Hence the rule for the real agent: every number must come from a tool, checked by code.

In [10]:
from src.agent.tools import run_tool

run_tool("get_inventory_policy", {"item": "FOODS_3_090", "policy": "(s, S)"})

{'item': 'FOODS_3_090',
 'policy': '(s, S)',
 'review_day': '2016-05-23',
 'demand_class': 'smooth',
 'price_usd': 1.6,
 'forecast_daily_units': 53.02,
 'rule': 'weekly review; order only when stock + on order <= s, then up to S',
 'z': 3.11,
 'reorder_point_s_units': 1263.3,
 'batch_eoq_units': 321.7,
 'order_up_to_S_units': 1585,
 'S_in_days_of_demand': 29.9,
 'S_stock_value_usd': 2536.0}